In [ ]:
!pip install -U bitsandbytes
!pip install -U langchain langchain-community langchain-core langchain-huggingface
!pip install datasets
!pip install -qU langchain-core langchain-upstage
!pip install -Uq langchain_community pymupdf faiss-cpu
!pip install -q transformers accelerate sentencepiece
!pip install -U langchain langchain-community langchain-core langchain-huggingface transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.1/500.1 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.0
    Uninstalling huggingface

# docs 불러오기

In [ ]:
# pubmed docs
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/pubmed_title_abstract.csv')
all_docs = df.to_dict(orient="records")



### 2.1 S-BERT

In [ ]:
!pip install gensim
!pip install transformers -U
!pip -q install -U chromadb langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 95.7 MB/s eta 0:00:00
  Using cached transformers-5.1.0-py3-none-any.whl.metadata (31 kB)
  Using cached huggingface_hub-1.4.1-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 136.9 MB/s eta 0:00:00
Using cached huggingface_hub-1.4.1-py3-none-any.whl (553 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 1.2.0 requires huggingface-hub<1.0.0,>=0.33.4, but you have huggingface-hu

In [ ]:
len(all_docs)

13007

In [ ]:
from sentence_transformers import SentenceTransformer, models
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from gensim.models.coherencemodel import CoherenceModel
import gensim.corpora as corpora
import os
import random
import numpy as np
import torch
from langchain.embeddings.base import Embeddings
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# (1) 모델 + Pooling 구성
word_embedding_model = models.Transformer("sentence-transformers/all-MiniLM-L6-v2")
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_cls_token = False,
    pooling_mode_max_tokens= True,
    pooling_mode_mean_tokens= True
    )
embedding_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

# (2) 임베딩 생성
docs = [d["text"] for d in all_docs]
#embeddings = embedding_model.encode(docs, batch_size=32, show_progress_bar=True, convert_to_numpy=True)

class MyCustomModelWrapper(Embeddings):
    def __init__(self, model):
        self.model = model  # 님이 만든 그 복잡한 모델을 여기에 담습니다.

    def embed_documents(self, texts):
        # Upstage처럼 문서 리스트를 받으면 알아서 벡터로 변환해주는 함수
        return self.model.encode(texts, batch_size=32, convert_to_numpy=True).tolist()

    def embed_query(self, text):
        # 질문 검색할 때 쓰이는 함수
        return self.model.encode([text], convert_to_numpy=True)[0].tolist()

# ------------------------------------------------------------------
# 2. "플러그" 장착! (이 객체는 이제 UpstageEmbeddings랑 똑같이 동작합니다)
# ------------------------------------------------------------------
# embedding_model은 사용자님이 위에서 만든 (Pooling 설정된) 그 모델입니다.
my_compatible_embeddings = MyCustomModelWrapper(embedding_model)


# ------------------------------------------------------------------
# 3. 이제 Upstage 코드랑 똑같은 방식으로 실행하면 됩니다!
# ------------------------------------------------------------------

# (1) 문서 리스트 만들기
document_list = [
    Document(
        page_content=d["text"],
        metadata={
            "pmid": str(d.get("pmid", "")),
            "title":  d.get("title", ""),
            "id":     d.get("id", i),        # 혹시 id 없으면 인덱스라도
        }
    )
    for i, d in enumerate(all_docs)
]
# (2) 텍스트 자르기 (Split)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
split_documents = text_splitter.split_documents(document_list)


##LLM

In [ ]:
# save_llama_embeddings.py

import os
import random
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer, models

# =====================================================
# 0) 환경 변수 & Seed 고정
# =====================================================

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


#MODEL_NAME = "Qwen/Qwen3-0.6B"
#MODEL_NAME ="BAAI/bge-m3"
#MODEL_NAME ="intfloat/multilingual-e5-large"
#MODEL_NAME ="OrdalieTech/Solon-embeddings-large-0.1"
#MODEL_NAME = "mixedbread-ai/mxbai-embed-large-v1"
MODEL_NAME = "google/embeddinggemma-300m"


# =====================================================
# 1) Llama 임베딩 레이어 로드
# =====================================================
word_embedding_model = models.Transformer(
    model_name_or_path= MODEL_NAME,
    model_args={
        "torch_dtype": torch.float32,
        "device_map": "auto",
        "trust_remote_code": True
    }
)
tokenizer = word_embedding_model.tokenizer
model = word_embedding_model.auto_model

# pad_token 없으면 eos를 pad로 사용(배치 패딩용)
if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        # eos조차 없으면 pad를 추가
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})
        model.resize_token_embeddings(len(tokenizer))

# config에도 pad_token_id 반영 (중요)
model.config.pad_token_id = tokenizer.pad_token_id


# =====================================================
# 2) 풀링 모듈 설정 (mean pooling)
# =====================================================
pooling_model = models.Pooling(
    word_embedding_dimension=word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_cls_token= False,
    pooling_mode_max_tokens= True,
    pooling_mode_mean_tokens= False
)

# =====================================================
# 3) SentenceTransformer 객체 생성
# =====================================================
embedding_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

# =====================================================
# 4) 문서 로드 및 임베딩 계산
# =====================================================
# df_all 파일 경로를 알맞게 수정하세요

docs = [d["text"] for d in all_docs]

class MyCustomModelWrapper(Embeddings):
    def __init__(self, model):
        self.model = model  # 님이 만든 그 복잡한 모델을 여기에 담습니다.

    def embed_documents(self, texts):
        # Upstage처럼 문서 리스트를 받으면 알아서 벡터로 변환해주는 함수
        return self.model.encode(texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True).tolist()

    def embed_query(self, text):
        # 질문 검색할 때 쓰이는 함수
        return self.model.encode([text], convert_to_numpy=True)[0].tolist()

# ------------------------------------------------------------------
# 2. "플러그" 장착! (이 객체는 이제 UpstageEmbeddings랑 똑같이 동작합니다)
# ------------------------------------------------------------------
# embedding_model은 사용자님이 위에서 만든 (Pooling 설정된) 그 모델입니다.
my_compatible_embeddings = MyCustomModelWrapper(embedding_model)


# ------------------------------------------------------------------
# 3. 이제 Upstage 코드랑 똑같은 방식으로 실행하면 됩니다!
# ------------------------------------------------------------------

# (1) 문서 리스트 만들기
document_list = [
    Document(
        page_content=d["text"],
        metadata={
            "pmid": str(d.get("pmid", "")),
            "title":  d.get("title", ""),
            "id":     d.get("id", i),        # 혹시 id 없으면 인덱스라도
        }
    )
    for i, d in enumerate(all_docs)
]
# (2) 텍스트 자르기 (Split)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
split_documents = text_splitter.split_documents(document_list)

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

### LLM2

In [ ]:
!pip install -q transformers accelerate sentencepiece
!huggingface-cli login

/bin/bash: line 1: huggingface-cli: command not found


In [ ]:
# save_embeddings_with_eos_pooling.py

import os, random
import numpy as np
import torch
from torch import nn

from sentence_transformers import SentenceTransformer, models

# (LangChain imports) - 버전 차이 대응
try:
    from langchain.embeddings.base import Embeddings
except Exception:
    from langchain_core.embeddings import Embeddings

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


# =====================================================
# 0) Seed
# =====================================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# =====================================================
# 1) 모델 로드
# =====================================================
MODEL_NAME = "meta-llama/Llama-3.2-1B"   # 예시: LLaMA
# MODEL_NAME = "BAAI/bge-m3"             # 참고: bge-m3는 보통 CLS가 정석이라 last-token은 비추

word_embedding_model = models.Transformer(
    model_name_or_path=MODEL_NAME,
    model_args={
        "torch_dtype": torch.float32,
        "device_map": "auto",
        "trust_remote_code": True,
    }
)

tokenizer = word_embedding_model.tokenizer
model = word_embedding_model.auto_model

# pad_token 없으면 eos를 pad로 사용(배치 패딩용)
if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        # eos조차 없으면 pad를 추가
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})
        model.resize_token_embeddings(len(tokenizer))

# config에도 pad_token_id 반영 (중요)
model.config.pad_token_id = tokenizer.pad_token_id


# =====================================================
# 2) "CLS 대신 EOS(last token)" 풀링 모듈
#    - attention_mask 기반으로 마지막 "실제 토큰"을 뽑음
#    - pad=eos여도 안전
# =====================================================
class MaskedConcatPooling(nn.Module):
    """
    use_last=True  -> 마지막 실제 토큰(last-token; EOS가 붙어있으면 사실상 EOS)
    use_mean=True  -> mean pooling (mask 적용)
    use_max=True   -> max pooling (mask 적용)
    결과는 (선택된 풀링들을) concat 해서 sentence_embedding으로 반환
    """
    def __init__(self, word_embedding_dimension: int, use_last=True, use_mean=False, use_max=False):
        super().__init__()
        self.word_embedding_dimension = word_embedding_dimension
        self.use_last = use_last
        self.use_mean = use_mean
        self.use_max = use_max

    def forward(self, features):
        token_embeddings = features["token_embeddings"]  # [B, T, H]
        attention_mask  = features["attention_mask"]     # [B, T]

        # [B, T, 1]
        mask = attention_mask.unsqueeze(-1).to(token_embeddings.dtype)

        outs = []

        # (A) last-token (마지막 실제 토큰)
        if self.use_last:
            last_idx = attention_mask.long().sum(dim=1) - 1     # [B]
            last_idx = torch.clamp(last_idx, min=0)
            batch_idx = torch.arange(token_embeddings.size(0), device=token_embeddings.device)
            last_vec = token_embeddings[batch_idx, last_idx]    # [B, H]
            outs.append(last_vec)

        # (B) mean pooling (mask 적용)
        if self.use_mean:
            summed = (token_embeddings * mask).sum(dim=1)        # [B, H]
            denom = mask.sum(dim=1).clamp(min=1e-6)              # [B, 1]
            mean_vec = summed / denom
            outs.append(mean_vec)

        # (C) max pooling (mask 적용)
        if self.use_max:
            # mask=0 위치는 매우 작은 값으로 보내서 max에서 제외
            minus_inf = torch.finfo(token_embeddings.dtype).min
            masked = token_embeddings.masked_fill(attention_mask.eq(0).unsqueeze(-1), minus_inf)
            max_vec = masked.max(dim=1).values                   # [B, H]
            outs.append(max_vec)

        if not outs:
            raise ValueError("At least one pooling must be enabled.")

        sentence_embedding = torch.cat(outs, dim=1) if len(outs) > 1 else outs[0]
        features["sentence_embedding"] = sentence_embedding
        return features

    def get_sentence_embedding_dimension(self):
        mult = int(self.use_last) + int(self.use_mean) + int(self.use_max)
        return self.word_embedding_dimension * mult


# =====================================================
# 3) SentenceTransformer 생성
#    - 여기서 "cls 대신 eos(last)" 쓰게 됨
# =====================================================
dim = word_embedding_model.get_word_embedding_dimension()

# ✅ CLS 대신 last-token 사용
# 필요하면 mean/max도 같이 켜서 듀얼/트리플 풀링 가능
pooling_model = MaskedConcatPooling(
    word_embedding_dimension=dim,
    use_last=False,      # ★ 핵심: CLS 대신 last-token(EOS/last)
    use_mean=True,      # 원하면 True
    use_max=True  # 원하면 True
)

embedding_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])


# =====================================================
# 4) (선택) 정말 EOS를 “끝에 확실히 붙이고” 싶으면
#    - last-token이 EOS가 되는 비율을 높임
#    - 다만 길이 제한(truncation) 걸리면 EOS가 잘릴 수 있음
# =====================================================
FORCE_APPEND_EOS = True
EOS = tokenizer.eos_token or ""

def maybe_append_eos(text: str) -> str:
    if not FORCE_APPEND_EOS or not EOS:
        return text
    text = "" if text is None else str(text)
    return text if text.endswith(EOS) else (text + EOS)


# =====================================================
# 5) LangChain 호환 wrapper
# =====================================================
class MyCustomModelWrapper(Embeddings):
    def __init__(self, model: SentenceTransformer, batch_size: int = 32, normalize: bool = False):
        self.model = model
        self.batch_size = batch_size
        self.normalize = normalize

    def embed_documents(self, texts):
        texts = [maybe_append_eos(t) for t in texts]
        vecs = self.model.encode(
            texts,
            batch_size=self.batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=self.normalize,
        )
        return vecs.tolist()

    def embed_query(self, text):
        text = maybe_append_eos(text)
        vec = self.model.encode(
            [text],
            convert_to_numpy=True,
            normalize_embeddings=self.normalize,
        )[0]
        return vec.tolist()


my_compatible_embeddings = MyCustomModelWrapper(embedding_model)


# =====================================================
# 6) 문서 split 예시 (네 all_docs 구조 반영)
# =====================================================
# all_docs = [{"text": "...", "pmid": "...", "title": "...", "id": ...}, ...]

document_list = [
    Document(
        page_content=d["text"],
        metadata={
            "pmid": str(d.get("pmid", "")),
            "title": d.get("title", ""),
            "id": d.get("id", i),
        },
    )
    for i, d in enumerate(all_docs)
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
split_documents = text_splitter.split_documents(document_list)

# split_documents를 Chroma에 넣을 때는 embedding_function=my_compatible_embeddings 사용하면 됨


## 3.rag 구현

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain.embeddings.base import Embeddings

### 3.2벡터DB**(VectorStore) 구축하기

In [ ]:
vectorstore = FAISS.from_documents(documents=split_documents,embedding=my_compatible_embeddings)
vectorstore.save_local("/content/drive/MyDrive/embedding3/llama_C_faiss")

In [ ]:
# (3) Chroma로 저장

from langchain_community.vectorstores import Chroma
import os

PERSIST_DIR = "/content/drive/MyDrive/chroma_db"
COLLECTION_NAME = "gemma_X"
os.makedirs(PERSIST_DIR, exist_ok=True)

vectorstore = Chroma.from_documents(
    documents=split_documents,
    embedding=my_compatible_embeddings,      # 너 wrapper 그대로
    persist_directory=PERSIST_DIR,
    collection_name=COLLECTION_NAME,
)

# (버전에 따라 필요)
try:
    vectorstore.persist()
except:
    pass

print("✅ saved:", PERSIST_DIR, "collection:", COLLECTION_NAME)


Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/171 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/92 [00:00<?, ?it/s]

✅ saved: /content/drive/MyDrive/chroma_db collection: gemma_X


In [ ]:
from chromadb import PersistentClient

client = PersistentClient(path="/content/drive/MyDrive/chroma_db")
print([c.name for c in client.list_collections()])


['gemma_M', 'gemma_X', 'gemma_C']
